In [1]:
"""
@author: Zilan Cheng
@note: This code is modified based on Zongyi Li's original implementation of Fourier Neural Operators.
"""

import torch.nn.functional as F
from timeit import default_timer
from utilities3 import *
import numpy as np
import matplotlib.pyplot as plt

torch.cuda.set_device(1)
torch.manual_seed(0)
np.random.seed(0)

In [2]:
class SpectralConv3d(nn.Module):
    def __init__(self, width, modes,basis):
        super(SpectralConv3d, self).__init__()

        """
        3D integration layer. It does POD transform, linear transform, and POD transform.    
        """

        self.width=width
        self.modes = modes
        self.basis = basis.to(torch.float32)[:,:,:,:self.modes].to(device)

        self.scale = (1 / (self.width*self.width))
        self.weights1 = nn.Parameter(self.scale * torch.rand(self.width, self.width, self.modes, dtype=torch.float))

    def forward(self, x):
        batchsize = x.shape[0]
        x_1  = torch.einsum("bwxyz,xyzf->bwf",x,self.basis).to(device) # f refers to coefficients space,  this step projects from physical space to coefficients space
        x_1 = torch.einsum("bif,iof->bof",x_1,self.weights1).to(device) # this step defines a diagonal operator 
        x_out  = torch.einsum("bwf,xyzf->bwxyz",x_1,self.basis).to(device) # f refers to coefficients space, this step recovers from coefficients space to physical space
        return x_out

In [3]:
class MLP(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels):
        super(MLP, self).__init__()
        self.mlp1 = nn.Conv3d(in_channels, mid_channels, 1)
        self.mlp2 = nn.Conv3d(mid_channels, out_channels, 1)

    def forward(self, x):
        x = self.mlp1(x)
        x = F.gelu(x)
        x = self.mlp2(x)
        return x

In [4]:
#Constant variables injected to biases
class MLP_t_theta(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels,t_channels,theta_channels):
        super(MLP_t_theta, self).__init__()
        self.mlp = nn.Conv3d(in_channels, out_channels, 1)
        self.theta_emb1 = nn.Linear(theta_channels, out_channels)
        self.theta_emb2 = nn.Linear(out_channels, out_channels)
        self.theta_act = F.gelu

    def forward(self, x, theta):
        x = self.mlp(x) 
        x = x + self.theta_emb2(self.theta_act(self.theta_emb1(theta)))[:, :, None, None, None] #
        return x

In [5]:
def get_grid(shape, device):
    batchsize, size_x, size_y = shape[0], shape[1], shape[2]
    gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
    gridx = gridx.reshape(1, size_x, 1, 1, 1).repeat([batchsize, 1, size_y, 2, 1])
    gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
    gridy = gridy.reshape(1, 1, size_y, 1, 1).repeat([batchsize, size_x, 1, 2, 1])
    return torch.cat((gridx, gridy), dim=-1).to(device)

In [6]:
class PODNO3d(nn.Module):
    def __init__(self, modes, width, basis, theta_channels = 1):
        super(PODNO3d, self).__init__()

        """
        The overall network. 
        1. Lift the input to the desire channel dimension.
        2. 4 integration layers.
        3. Project back to the physical space.
        
        input: (a(x, y), epsilon)
        output: u(x,y)
        """

        self.modes = modes
        self.width = width
        self.basis = torch.tensor(basis)
        x_channels = 3
        t_channels =1

        self.p = nn.Linear(3, self.width) # input channel is 3: (a(x, y), x, y)
        self.conv0 = SpectralConv3d(self.width, self.modes, self.basis)
        self.conv1 = SpectralConv3d(self.width, self.modes, self.basis)
        self.conv2 = SpectralConv3d(self.width, self.modes, self.basis)
        self.conv3 = SpectralConv3d(self.width, self.modes, self.basis)
        self.mlp0 = MLP_t_theta(self.width, self.width, self.width,t_channels,theta_channels)
        self.mlp1 = MLP_t_theta(self.width, self.width, self.width,t_channels,theta_channels)
        self.mlp2 = MLP_t_theta(self.width, self.width, self.width,t_channels,theta_channels)
        self.mlp3 = MLP_t_theta(self.width, self.width, self.width,t_channels,theta_channels)
        self.w0 = nn.Conv3d(self.width, self.width, 1)
        self.w1 = nn.Conv3d(self.width, self.width, 1)
        self.w2 = nn.Conv3d(self.width, self.width, 1)
        self.w3 = nn.Conv3d(self.width, self.width, 1)
        self.q = MLP(self.width, 1, self.width * 4) # output channel is 1: u(x, y)

    def forward(self, x,theta):
        grid = get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x=x.to(torch.float32)
        x = self.p(x)
        x = x.permute(0, 4, 1, 2, 3)
        x1 = self.conv0(x)
        x1 = self.mlp0(x1,theta)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x1 = self.mlp1(x1,theta)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x1 = self.mlp2(x1,theta)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x1 = self.mlp3(x1,theta)
        x2 = self.w3(x)
        x = x1 + x2

        x = self.q(x)
        x = x.permute(0, 2, 3, 4, 1)
        return x

In [7]:
################################################################
# configs
################################################################

ntrain = 900
ntest = 100
nsnap =900

width = 32
modes =288

mesh=64
r=1

batch_size = 20
learning_rate = 0.001
epochs = 500
iterations = epochs*(ntrain//batch_size)

In [8]:
################################################################
# dataloader
################################################################
dataloader=MatReader('../data/nls/pNLS_data.mat')

u_re1=dataloader.read_field('u_re1')
u_im1=dataloader.read_field('u_im1')
u_re2=dataloader.read_field('u_re2')
u_im2=dataloader.read_field('u_im2')
theta_train=dataloader.read_field('p1')[:,:ntrain].permute(1,0)
theta_test=dataloader.read_field('p1')[:,-ntest:].permute(1,0)

u0_train_re=u_re1[:,:,:ntrain][::r,::r,:]
u0_train_im=u_im1[:,:,:ntrain][::r,::r,:]
u_end_train_re=u_re2[:,:,:ntrain][::r,::r,:]
u_end_train_im=u_im2[:,:,:ntrain][::r,::r,:]
u0_test_re=u_re1[:,:,-ntest:][::r,::r,:]
u0_test_im=u_im1[:,:,-ntest:][::r,::r,:]
u_end_test_re=u_re2[:,:,-ntest:][::r,::r,:]
u_end_test_im=u_im2[:,:,-ntest:][::r,::r,:]

u_train_0=torch.stack((u0_train_re,u0_train_im),dim=0)
u_test_0=torch.stack((u0_test_re,u0_test_im),dim=0)
u_train_end=torch.stack((u_end_train_re,u_end_train_im),dim=0)
u_test_end=torch.stack((u_end_test_re,u_end_test_im),dim=0)

x_train=u_train_0.permute(3,1,2,0)
x_test=u_test_0.permute(3,1,2,0)
y_train=u_train_end.permute(3,1,2,0)
y_test=u_test_end.permute(3,1,2,0)

x_train = x_train.reshape(ntrain,mesh,mesh,2,1)
x_test = x_test.reshape(ntest,mesh,mesh,2,1)
y_train = y_train.reshape(ntrain,mesh,mesh,2,1)
y_test = y_test.reshape(ntest,mesh,mesh,2,1)

x_normalizer = UnitGaussianNormalizer(x_train)
x_train = x_normalizer.encode(x_train)
x_test = x_normalizer.encode(x_test)

theta_normalizer = UnitGaussianNormalizer(theta_train)
theta_train = theta_normalizer.encode(theta_train)
theta_test = theta_normalizer.encode(theta_test)

y_normalizer = UnitGaussianNormalizer(y_train)
y_train = y_normalizer.encode(y_train)

train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train, theta_train,y_train), batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, theta_test, y_test), batch_size=batch_size, shuffle=False)

d1=mesh
d2=mesh
d3=2

In [9]:
################################################################
# SVD
################################################################
x_snap = x_train[:nsnap,:,:,:,:]
y_snap =y_train[:nsnap,:,:,:,:]

basis_0 = torch.cat((x_snap, y_snap), dim=0)
basis_1=basis_0.permute(1,2,3,4,0)
basis_1=basis_1.reshape(-1,ntrain*2)
U,S,V=torch.svd(torch.mm(basis_1,basis_1.T))
basis=U.reshape(mesh,mesh,2,mesh*mesh*2)

In [10]:
solution_real=torch.zeros(ntest,mesh,mesh,2,1)
solution_learnt=torch.zeros(ntest,mesh,mesh,2,1)

In [11]:
################################################################
# training and evaluation
################################################################
model = PODNO3d(modes,width,basis,theta_channels = 1).to(device)
print(count_params(model))

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=iterations)

myloss = LpLoss(size_average=True)
y_normalizer.to(device)
for ep in range(epochs):
    i=0
    model.train()
    train_l2 = 0
    for x, theta,  y in train_loader:
        x, theta,  y = x.to(device), theta.to(device),y.to(device)

        optimizer.zero_grad()
        out = model(x,theta).reshape(batch_size, d1, d2, d3,1)
        out = y_normalizer.decode(out)
        y = y_normalizer.decode(y)

        loss =myloss(out.reshape(batch_size,d1*d2*d3),y.reshape(batch_size,d1*d2*d3))
        loss.backward()

        optimizer.step()
        scheduler.step()
        train_l2 += loss.item()

    model.eval()
    test_l2 = 0.0
    with torch.no_grad():
        for x, theta, y in test_loader:
            x, theta, y = x.to(device), theta.to(device),y.to(device)

            out = model(x,theta).reshape(batch_size, d1,d2,d3,1)
            out = y_normalizer.decode(out)
            solution_real[i:i+batch_size,:,:,:,:]=y
            solution_learnt[i:i+batch_size,:,:,:,:]=out
            i=i+batch_size
            test_l2 += myloss(out.view(batch_size,d1*d2*d3), y.view(batch_size,d1*d2*d3)).item()
    train_l2/= ntrain/batch_size
    test_l2 /= ntest/batch_size

    if ep % 50 == 0 or ep == epochs - 1:
        print(f"Epoch {ep:4d} | Train L2: {train_l2:.6f} | Test L2: {test_l2:.6f}")


/tmp/ipykernel_3163750/2219388632.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.basis = torch.tensor(basis)


1197057
Epoch    0 | Train L2: 0.641100 | Test L2: 0.517684
Epoch   50 | Train L2: 0.085890 | Test L2: 0.089860
Epoch  100 | Train L2: 0.070829 | Test L2: 0.081374
Epoch  150 | Train L2: 0.060742 | Test L2: 0.077366
Epoch  200 | Train L2: 0.051754 | Test L2: 0.073228
Epoch  250 | Train L2: 0.045631 | Test L2: 0.070511
Epoch  300 | Train L2: 0.039433 | Test L2: 0.068595
Epoch  350 | Train L2: 0.033658 | Test L2: 0.066690
Epoch  400 | Train L2: 0.029779 | Test L2: 0.065689
Epoch  450 | Train L2: 0.027418 | Test L2: 0.065237
Epoch  499 | Train L2: 0.026709 | Test L2: 0.065047
